# Ridge Regression - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import Ridge as SklearnRidge, RidgeCV
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Ridge Regression?

Ridge Regression (also known as Tikhonov regularization) is a **regularized** version of linear regression that adds an **L2 penalty** to the loss function. This penalty shrinks the coefficients toward zero, preventing overfitting.

### Mathematical Formulation

#### Ordinary Least Squares (OLS) Loss
$$J_{OLS}(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2 = \frac{1}{2m} ||X\theta - y||_2^2$$

#### Ridge Regression Loss (with L2 Penalty)
$$J_{Ridge}(\theta) = \frac{1}{2m} ||X\theta - y||_2^2 + \alpha ||\theta||_2^2$$

Where:
- $\alpha$ (or $\lambda$) is the **regularization parameter**
- $||\theta||_2^2 = \sum_{j=1}^{n} \theta_j^2$ is the **L2 norm squared** (ridge penalty)
- Note: The bias term $\theta_0$ is typically NOT regularized

#### Closed-Form Solution
Unlike Lasso, Ridge regression has a **closed-form solution**:
$$\hat{\theta} = (X^TX + \alpha I)^{-1}X^Ty$$

Where $I$ is the identity matrix (with a 0 in the position corresponding to the bias if not regularizing it).

### Why L2 Regularization Works

1. **Adds to the diagonal** of $X^TX$, making it always invertible
2. **Shrinks coefficients** proportionally to their magnitude
3. **Never sets coefficients to exactly zero** (unlike L1/Lasso)
4. **Handles multicollinearity** by distributing weight among correlated features

### Bias-Variance Tradeoff

| Alpha Value | Bias | Variance | Model Complexity |
|-------------|------|----------|------------------|
| $\alpha = 0$ | Low | High | High (OLS) |
| $\alpha$ small | Low-Medium | Medium-High | Medium-High |
| $\alpha$ optimal | Balanced | Balanced | Optimal |
| $\alpha$ large | High | Low | Low |
| $\alpha \to \infty$ | High | Very Low | All coefficients near 0 |

### Alpha Selection Strategies

1. **Cross-Validation**: Most common - find alpha that minimizes CV error
2. **Grid Search**: Try many alpha values on log scale (e.g., $10^{-4}$ to $10^4$)
3. **Information Criteria**: AIC, BIC for model selection
4. **L-curve Method**: Plot residual norm vs. solution norm

### Time Complexity
- Training (closed-form): $O(n^2m + n^3)$ where n = features, m = samples
- Training (gradient descent): $O(nm \cdot iterations)$
- Prediction: $O(nm)$

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class RidgeRegressionScratch:
    """
    Ridge Regression implementation from scratch using closed-form solution.
    
    Parameters:
    -----------
    alpha : float, default=1.0
        Regularization strength. Larger values specify stronger regularization.
        Also known as lambda in some literature.
    fit_intercept : bool, default=True
        Whether to calculate the intercept for this model.
    normalize_features : bool, default=False
        If True, features will be normalized before fitting.
        Note: sklearn's Ridge uses StandardScaler separately.
    """
    
    def __init__(self, alpha=1.0, fit_intercept=True, normalize_features=False):
        self.alpha = alpha
        self.fit_intercept = fit_intercept
        self.normalize_features = normalize_features
        self.weights = None
        self.intercept_ = 0.0
        self.coef_ = None
        self._feature_mean = None
        self._feature_std = None
        self._y_mean = None
        
    def _normalize(self, X):
        """
        Normalize features to zero mean and unit variance.
        """
        return (X - self._feature_mean) / (self._feature_std + 1e-8)
    
    def fit(self, X, y):
        """
        Fit Ridge regression model using closed-form solution.
        
        The closed-form solution is:
        theta = (X^T X + alpha * I)^(-1) X^T y
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target values
            
        Returns:
        --------
        self : object
            Returns self.
        """
        # Convert to numpy arrays
        X = np.array(X, dtype=np.float64)
        y = np.array(y, dtype=np.float64)
        
        n_samples, n_features = X.shape
        
        # Store normalization parameters if needed
        if self.normalize_features:
            self._feature_mean = X.mean(axis=0)
            self._feature_std = X.std(axis=0)
            X = self._normalize(X)
        
        # Handle intercept by centering
        if self.fit_intercept:
            self._y_mean = y.mean()
            self._X_mean = X.mean(axis=0)
            X_centered = X - self._X_mean
            y_centered = y - self._y_mean
        else:
            X_centered = X
            y_centered = y
        
        # Create identity matrix for regularization
        # We don't regularize the bias term (handled by centering)
        identity = np.eye(n_features)
        
        # Closed-form solution: theta = (X^T X + alpha * I)^(-1) X^T y
        # Using more numerically stable approach with solve instead of inverse
        XtX = X_centered.T @ X_centered
        Xty = X_centered.T @ y_centered
        
        # Add regularization term
        regularized_matrix = XtX + self.alpha * identity
        
        # Solve the linear system (more stable than computing inverse)
        self.coef_ = np.linalg.solve(regularized_matrix, Xty)
        
        # Calculate intercept
        if self.fit_intercept:
            self.intercept_ = self._y_mean - self._X_mean @ self.coef_
        else:
            self.intercept_ = 0.0
            
        return self
    
    def predict(self, X):
        """
        Predict using the Ridge regression model.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Samples to predict.
            
        Returns:
        --------
        y_pred : array, shape (n_samples,)
            Predicted values.
        """
        X = np.array(X, dtype=np.float64)
        
        if self.normalize_features:
            X = self._normalize(X)
            
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        """
        Return the coefficient of determination R^2 of the prediction.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Test samples.
        y : array-like, shape (n_samples,)
            True values for X.
            
        Returns:
        --------
        score : float
            R^2 score.
        """
        y_pred = self.predict(X)
        y = np.array(y, dtype=np.float64)
        
        # Total sum of squares
        ss_tot = np.sum((y - y.mean()) ** 2)
        # Residual sum of squares
        ss_res = np.sum((y - y_pred) ** 2)
        
        return 1 - (ss_res / ss_tot)
    
    def get_params(self):
        """
        Get model parameters.
        """
        return {
            'alpha': self.alpha,
            'fit_intercept': self.fit_intercept,
            'normalize_features': self.normalize_features
        }

In [ ]:
# Test the implementation with a simple example
print("Testing RidgeRegressionScratch with synthetic data...")
print("="*50)

# Create simple synthetic data
np.random.seed(42)
X_simple = np.random.randn(100, 3)
true_coef = np.array([2.0, -1.5, 0.5])
y_simple = X_simple @ true_coef + 1.0 + np.random.randn(100) * 0.5

# Fit our model
ridge_test = RidgeRegressionScratch(alpha=1.0, fit_intercept=True)
ridge_test.fit(X_simple, y_simple)

print(f"True coefficients: {true_coef}")
print(f"Estimated coefficients: {ridge_test.coef_}")
print(f"True intercept: 1.0")
print(f"Estimated intercept: {ridge_test.intercept_:.4f}")
print(f"R^2 Score: {ridge_test.score(X_simple, y_simple):.4f}")

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Load California Housing dataset (using a subset for speed)
print("Loading California Housing dataset...")
california = load_california_housing()
X_full, y_full = california.data, california.target
feature_names = california.feature_names

print(f"Full dataset shape: {X_full.shape}")
print(f"Features: {feature_names}")

# Use a subset for faster computation (5000 samples)
subset_size = 5000
indices = np.random.choice(len(X_full), subset_size, replace=False)
X = X_full[indices]
y = y_full[indices]

print(f"\nUsing subset of {subset_size} samples for efficient CPU computation")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features (important for Ridge regression!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}] (in $100,000s)")

In [ ]:
# Train Ridge models with different alpha values
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
results = []

print("Training Ridge Regression with various alpha values...")
print("="*70)
print(f"{'Alpha':<10} {'Train R2':>12} {'Test R2':>12} {'Train MSE':>12} {'Test MSE':>12}")
print("-"*70)

for alpha in alphas:
    model = RidgeRegressionScratch(alpha=alpha, fit_intercept=True)
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # Metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    results.append({
        'alpha': alpha,
        'model': model,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'coef': model.coef_.copy()
    })
    
    print(f"{alpha:<10.3f} {train_r2:>12.4f} {test_r2:>12.4f} {train_mse:>12.4f} {test_mse:>12.4f}")

# Convert to DataFrame for easy analysis
results_df = pd.DataFrame(results)

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
# MSE vs Alpha plot - Finding optimal regularization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test with more alpha values for smoother curves
alphas_fine = np.logspace(-4, 4, 50)
train_mses = []
test_mses = []

for alpha in alphas_fine:
    model = RidgeRegressionScratch(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    train_mses.append(mean_squared_error(y_train, model.predict(X_train_scaled)))
    test_mses.append(mean_squared_error(y_test, model.predict(X_test_scaled)))

# Plot MSE vs Alpha
axes[0].semilogx(alphas_fine, train_mses, 'b-', label='Train MSE', linewidth=2)
axes[0].semilogx(alphas_fine, test_mses, 'r-', label='Test MSE', linewidth=2)
axes[0].axvline(alphas_fine[np.argmin(test_mses)], color='green', linestyle='--', 
                label=f'Optimal alpha = {alphas_fine[np.argmin(test_mses)]:.4f}')
axes[0].set_xlabel('Alpha (log scale)', fontsize=12)
axes[0].set_ylabel('Mean Squared Error', fontsize=12)
axes[0].set_title('MSE vs Regularization Strength (Alpha)', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# R^2 vs Alpha
train_r2s = []
test_r2s = []

for alpha in alphas_fine:
    model = RidgeRegressionScratch(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    train_r2s.append(model.score(X_train_scaled, y_train))
    test_r2s.append(model.score(X_test_scaled, y_test))

axes[1].semilogx(alphas_fine, train_r2s, 'b-', label='Train R2', linewidth=2)
axes[1].semilogx(alphas_fine, test_r2s, 'r-', label='Test R2', linewidth=2)
axes[1].axvline(alphas_fine[np.argmax(test_r2s)], color='green', linestyle='--',
                label=f'Optimal alpha = {alphas_fine[np.argmax(test_r2s)]:.4f}')
axes[1].set_xlabel('Alpha (log scale)', fontsize=12)
axes[1].set_ylabel('R-squared Score', fontsize=12)
axes[1].set_title('R-squared vs Regularization Strength (Alpha)', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Optimal alpha (minimum test MSE): {alphas_fine[np.argmin(test_mses)]:.4f}")
print(f"Minimum test MSE: {min(test_mses):.4f}")

In [ ]:
# Coefficient Paths - How coefficients change with alpha
alphas_path = np.logspace(-4, 4, 100)
coef_paths = []

for alpha in alphas_path:
    model = RidgeRegressionScratch(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    coef_paths.append(model.coef_.copy())

coef_paths = np.array(coef_paths)

# Plot coefficient paths
plt.figure(figsize=(12, 6))
for i, name in enumerate(feature_names):
    plt.semilogx(alphas_path, coef_paths[:, i], label=name, linewidth=2)

plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('Alpha (log scale)', fontsize=12)
plt.ylabel('Coefficient Value', fontsize=12)
plt.title('Ridge Regression Coefficient Paths (Regularization Path)', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-Validation for Alpha Selection
print("Performing K-Fold Cross-Validation for Alpha Selection...")
print("="*60)

alphas_cv = np.logspace(-3, 3, 30)
cv_scores_mean = []
cv_scores_std = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for alpha in alphas_cv:
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X_train_scaled):
        X_fold_train = X_train_scaled[train_idx]
        X_fold_val = X_train_scaled[val_idx]
        y_fold_train = y_train[train_idx]
        y_fold_val = y_train[val_idx]
        
        model = RidgeRegressionScratch(alpha=alpha)
        model.fit(X_fold_train, y_fold_train)
        fold_scores.append(model.score(X_fold_val, y_fold_val))
    
    cv_scores_mean.append(np.mean(fold_scores))
    cv_scores_std.append(np.std(fold_scores))

cv_scores_mean = np.array(cv_scores_mean)
cv_scores_std = np.array(cv_scores_std)

# Find best alpha
best_alpha_idx = np.argmax(cv_scores_mean)
best_alpha = alphas_cv[best_alpha_idx]

# Plot CV results
plt.figure(figsize=(10, 6))
plt.semilogx(alphas_cv, cv_scores_mean, 'b-', linewidth=2, label='Mean CV Score')
plt.fill_between(alphas_cv, 
                 cv_scores_mean - cv_scores_std,
                 cv_scores_mean + cv_scores_std,
                 alpha=0.3, label='Std Dev')
plt.axvline(best_alpha, color='red', linestyle='--', 
            label=f'Best alpha = {best_alpha:.4f}')
plt.xlabel('Alpha (log scale)', fontsize=12)
plt.ylabel('CV R-squared Score', fontsize=12)
plt.title('5-Fold Cross-Validation for Alpha Selection', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nBest Alpha from Cross-Validation: {best_alpha:.4f}")
print(f"Best CV Score: {cv_scores_mean[best_alpha_idx]:.4f} (+/- {cv_scores_std[best_alpha_idx]:.4f})")

In [ ]:
# Train final model with best alpha
print("Training Final Model with Optimal Alpha...")
print("="*50)

final_model = RidgeRegressionScratch(alpha=best_alpha)
final_model.fit(X_train_scaled, y_train)

# Final predictions
y_train_pred = final_model.predict(X_train_scaled)
y_test_pred = final_model.predict(X_test_scaled)

# Comprehensive metrics
print(f"\nFinal Model Performance (alpha={best_alpha:.4f}):")
print("-"*50)
print(f"Training Metrics:")
print(f"  R-squared: {r2_score(y_train, y_train_pred):.4f}")
print(f"  MSE: {mean_squared_error(y_train, y_train_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")

print(f"\nTest Metrics:")
print(f"  R-squared: {r2_score(y_test, y_test_pred):.4f}")
print(f"  MSE: {mean_squared_error(y_test, y_test_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")

# Feature importance (coefficient magnitudes)
print(f"\nFeature Coefficients (absolute values indicate importance):")
print("-"*50)
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': final_model.coef_,
    'Abs Coefficient': np.abs(final_model.coef_)
}).sort_values('Abs Coefficient', ascending=False)

print(coef_df.to_string(index=False))

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
# Coefficient Shrinkage Comparison Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train OLS (alpha=0) and Ridge with high alpha
ols_model = RidgeRegressionScratch(alpha=0.0001)  # Near-zero alpha = OLS
ols_model.fit(X_train_scaled, y_train)

ridge_strong = RidgeRegressionScratch(alpha=100.0)
ridge_strong.fit(X_train_scaled, y_train)

# Bar plot comparison
x_pos = np.arange(len(feature_names))
width = 0.35

axes[0].bar(x_pos - width/2, ols_model.coef_, width, label='OLS (alpha~0)', color='blue', alpha=0.7)
axes[0].bar(x_pos + width/2, final_model.coef_, width, label=f'Ridge (alpha={best_alpha:.2f})', color='red', alpha=0.7)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(feature_names, rotation=45, ha='right')
axes[0].set_ylabel('Coefficient Value', fontsize=12)
axes[0].set_title('Coefficient Shrinkage: OLS vs Ridge', fontsize=14)
axes[0].legend()
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].grid(True, alpha=0.3, axis='y')

# Coefficient magnitude by regularization strength
alphas_shrink = [0.0001, 0.01, 0.1, 1.0, 10.0, 100.0]
coef_norms = []

for alpha in alphas_shrink:
    model = RidgeRegressionScratch(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    coef_norms.append(np.linalg.norm(model.coef_))

axes[1].semilogx(alphas_shrink, coef_norms, 'go-', linewidth=2, markersize=10)
axes[1].set_xlabel('Alpha (log scale)', fontsize=12)
axes[1].set_ylabel('L2 Norm of Coefficients', fontsize=12)
axes[1].set_title('Coefficient Shrinkage Effect', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.5, edgecolors='none', s=30)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Values', fontsize=12)
axes[0].set_ylabel('Predicted Values', fontsize=12)
axes[0].set_title(f'Training Set: Predicted vs Actual\nR2 = {r2_score(y_train, y_train_pred):.4f}', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, edgecolors='none', s=30, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Values', fontsize=12)
axes[1].set_ylabel('Predicted Values', fontsize=12)
axes[1].set_title(f'Test Set: Predicted vs Actual\nR2 = {r2_score(y_test, y_test_pred):.4f}', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

# Residuals vs Predicted (Train)
axes[0, 0].scatter(y_train_pred, residuals_train, alpha=0.5, edgecolors='none', s=30)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Values', fontsize=12)
axes[0, 0].set_ylabel('Residuals', fontsize=12)
axes[0, 0].set_title('Training Set: Residuals vs Predicted', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)

# Residuals vs Predicted (Test)
axes[0, 1].scatter(y_test_pred, residuals_test, alpha=0.5, edgecolors='none', s=30, color='green')
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted Values', fontsize=12)
axes[0, 1].set_ylabel('Residuals', fontsize=12)
axes[0, 1].set_title('Test Set: Residuals vs Predicted', fontsize=14)
axes[0, 1].grid(True, alpha=0.3)

# Residual Distribution (Train)
axes[1, 0].hist(residuals_train, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Residual Value', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title(f'Training Residual Distribution\nMean={residuals_train.mean():.4f}, Std={residuals_train.std():.4f}', fontsize=14)
axes[1, 0].grid(True, alpha=0.3)

# Residual Distribution (Test)
axes[1, 1].hist(residuals_test, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Residual Value', fontsize=12)
axes[1, 1].set_ylabel('Frequency', fontsize=12)
axes[1, 1].set_title(f'Test Residual Distribution\nMean={residuals_test.mean():.4f}, Std={residuals_test.std():.4f}', fontsize=14)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sorted coefficients
sorted_idx = np.argsort(np.abs(final_model.coef_))[::-1]
sorted_features = [feature_names[i] for i in sorted_idx]
sorted_coefs = final_model.coef_[sorted_idx]

colors = ['green' if c > 0 else 'red' for c in sorted_coefs]

axes[0].barh(range(len(sorted_features)), sorted_coefs, color=colors, alpha=0.7)
axes[0].set_yticks(range(len(sorted_features)))
axes[0].set_yticklabels(sorted_features)
axes[0].set_xlabel('Coefficient Value', fontsize=12)
axes[0].set_title('Feature Coefficients (Sorted by Magnitude)', fontsize=14)
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
axes[0].grid(True, alpha=0.3, axis='x')

# Absolute values
axes[1].barh(range(len(sorted_features)), np.abs(sorted_coefs), color='steelblue', alpha=0.7)
axes[1].set_yticks(range(len(sorted_features)))
axes[1].set_yticklabels(sorted_features)
axes[1].set_xlabel('Absolute Coefficient Value', fontsize=12)
axes[1].set_title('Feature Importance (|Coefficient|)', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Ridge Regression

#### Ideal Use Cases:

1. **Multicollinearity Present**
   - When features are highly correlated
   - Ridge distributes weight among correlated features instead of arbitrarily picking one
   - Example: Predicting with both "square feet" and "number of rooms"

2. **Many Features Relative to Samples**
   - When p (features) approaches or exceeds n (samples)
   - Prevents overfitting in high-dimensional spaces
   - Example: Genomics data with thousands of genes

3. **All Features Potentially Relevant**
   - When you believe all features contribute to the prediction
   - Ridge keeps all features (shrinks but doesn't eliminate)
   - Example: Economic indicators predicting GDP

4. **Numerical Stability Needed**
   - When $X^TX$ is near-singular
   - Adding $\alpha I$ ensures invertibility
   - Example: Ill-conditioned design matrices

5. **Continuous Predictions**
   - Regression problems (not classification)
   - Real-valued outputs

### When NOT to Use Ridge Regression

#### Avoid Ridge When:

1. **Sparse Solutions Needed**
   - Ridge never sets coefficients to exactly zero
   - Use Lasso (L1) or Elastic Net instead
   - Example: Feature selection is the primary goal

2. **Interpretability with Feature Selection**
   - If you need to identify the "important" features
   - Ridge keeps all features, making interpretation harder
   - Use Lasso for automatic feature selection

3. **Non-linear Relationships**
   - Ridge assumes linear relationships
   - Use polynomial features, kernel methods, or tree-based models

4. **Outlier-Heavy Data**
   - MSE loss is sensitive to outliers
   - Consider robust regression methods

5. **Classification Tasks**
   - Use Logistic Regression with L2 penalty instead

### Alpha Selection Strategies

| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| **Cross-Validation** | Data-driven, reliable | Computationally expensive | Most cases |
| **Grid Search** | Simple to implement | Needs good range | Quick exploration |
| **RidgeCV** (sklearn) | Efficient LOO-CV | Limited to built-in | Production use |
| **Bayesian Methods** | Principled, uncertainty | Complex implementation | Research |
| **L-Curve** | Visual intuition | Subjective choice | Understanding data |

### Practical Tips

1. **Always scale features** before applying Ridge
2. **Start with alpha search** in range $[10^{-4}, 10^{4}]$
3. **Use log scale** for alpha search
4. **Monitor both train and test** metrics
5. **Consider Elastic Net** if unsure between Ridge and Lasso

### Comparison Table: Ridge vs Lasso vs Elastic Net

| Feature | Ridge (L2) | Lasso (L1) | Elastic Net |
|---------|------------|------------|-------------|
| Penalty | $\alpha\sum\theta_j^2$ | $\alpha\sum|\theta_j|$ | Both L1 + L2 |
| Feature Selection | No | Yes | Yes |
| Multicollinearity | Handles well | Arbitrary selection | Handles well |
| Closed-form | Yes | No | No |
| Grouped Features | Keeps all | Picks one | Can keep groups |

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn's Ridge
print("Comparison: Our Implementation vs sklearn Ridge")
print("="*60)

# Use same alpha for fair comparison
test_alpha = best_alpha

# Our implementation
our_ridge = RidgeRegressionScratch(alpha=test_alpha)
our_ridge.fit(X_train_scaled, y_train)

# sklearn implementation
sklearn_ridge = SklearnRidge(alpha=test_alpha, fit_intercept=True)
sklearn_ridge.fit(X_train_scaled, y_train)

# Predictions
our_pred_train = our_ridge.predict(X_train_scaled)
our_pred_test = our_ridge.predict(X_test_scaled)

sklearn_pred_train = sklearn_ridge.predict(X_train_scaled)
sklearn_pred_test = sklearn_ridge.predict(X_test_scaled)

# Compare metrics
print(f"\nAlpha used: {test_alpha:.4f}")
print("-"*60)

print("\nOur Implementation:")
print(f"  Train R2: {r2_score(y_train, our_pred_train):.6f}")
print(f"  Test R2: {r2_score(y_test, our_pred_test):.6f}")
print(f"  Train MSE: {mean_squared_error(y_train, our_pred_train):.6f}")
print(f"  Test MSE: {mean_squared_error(y_test, our_pred_test):.6f}")

print("\nsklearn Implementation:")
print(f"  Train R2: {r2_score(y_train, sklearn_pred_train):.6f}")
print(f"  Test R2: {r2_score(y_test, sklearn_pred_test):.6f}")
print(f"  Train MSE: {mean_squared_error(y_train, sklearn_pred_train):.6f}")
print(f"  Test MSE: {mean_squared_error(y_test, sklearn_pred_test):.6f}")

# Compare coefficients
print("\nCoefficient Comparison:")
print("-"*60)
coef_comparison = pd.DataFrame({
    'Feature': feature_names,
    'Our Coef': our_ridge.coef_,
    'sklearn Coef': sklearn_ridge.coef_,
    'Difference': our_ridge.coef_ - sklearn_ridge.coef_
})
print(coef_comparison.to_string(index=False))

print(f"\nIntercept Comparison:")
print(f"  Our intercept: {our_ridge.intercept_:.6f}")
print(f"  sklearn intercept: {sklearn_ridge.intercept_:.6f}")
print(f"  Difference: {our_ridge.intercept_ - sklearn_ridge.intercept_:.8f}")

In [ ]:
# Visual comparison of predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Our vs sklearn predictions
axes[0].scatter(sklearn_pred_test, our_pred_test, alpha=0.5, edgecolors='none', s=30)
min_val = min(sklearn_pred_test.min(), our_pred_test.min())
max_val = max(sklearn_pred_test.max(), our_pred_test.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Agreement')
axes[0].set_xlabel('sklearn Predictions', fontsize=12)
axes[0].set_ylabel('Our Predictions', fontsize=12)
axes[0].set_title('Prediction Comparison: Our vs sklearn', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Coefficient comparison bar plot
x_pos = np.arange(len(feature_names))
width = 0.35

axes[1].bar(x_pos - width/2, our_ridge.coef_, width, label='Our Implementation', alpha=0.7)
axes[1].bar(x_pos + width/2, sklearn_ridge.coef_, width, label='sklearn', alpha=0.7)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(feature_names, rotation=45, ha='right')
axes[1].set_ylabel('Coefficient Value', fontsize=12)
axes[1].set_title('Coefficient Comparison', fontsize=14)
axes[1].legend()
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Quantitative agreement metrics
pred_correlation = np.corrcoef(sklearn_pred_test, our_pred_test)[0, 1]
coef_correlation = np.corrcoef(sklearn_ridge.coef_, our_ridge.coef_)[0, 1]
max_pred_diff = np.max(np.abs(sklearn_pred_test - our_pred_test))
max_coef_diff = np.max(np.abs(sklearn_ridge.coef_ - our_ridge.coef_))

print(f"\nAgreement Metrics:")
print(f"  Prediction Correlation: {pred_correlation:.8f}")
print(f"  Coefficient Correlation: {coef_correlation:.8f}")
print(f"  Max Prediction Difference: {max_pred_diff:.8f}")
print(f"  Max Coefficient Difference: {max_coef_diff:.8f}")

In [ ]:
# Compare with sklearn's RidgeCV for alpha selection
print("Comparing Alpha Selection: Our CV vs sklearn RidgeCV")
print("="*60)

# sklearn RidgeCV
alphas_for_cv = np.logspace(-3, 3, 30)
sklearn_ridgecv = RidgeCV(alphas=alphas_for_cv, cv=5)
sklearn_ridgecv.fit(X_train_scaled, y_train)

print(f"Our CV selected alpha: {best_alpha:.4f}")
print(f"sklearn RidgeCV selected alpha: {sklearn_ridgecv.alpha_:.4f}")

# Compare final performance
print(f"\nPerformance with respective optimal alphas:")
print("-"*60)

# Our model with our CV alpha
our_final = RidgeRegressionScratch(alpha=best_alpha)
our_final.fit(X_train_scaled, y_train)

print(f"Our Implementation (alpha={best_alpha:.4f}):")
print(f"  Test R2: {our_final.score(X_test_scaled, y_test):.6f}")

print(f"\nsklearn RidgeCV (alpha={sklearn_ridgecv.alpha_:.4f}):")
print(f"  Test R2: {sklearn_ridgecv.score(X_test_scaled, y_test):.6f}")

## Summary & Key Takeaways

### What We Learned:

1. **Mathematical Foundation**: Ridge regression adds an L2 penalty to prevent overfitting and handle multicollinearity
2. **Closed-Form Solution**: Unlike Lasso, Ridge has an efficient analytical solution
3. **Alpha Selection**: Cross-validation is the gold standard for finding optimal regularization strength
4. **Coefficient Behavior**: Ridge shrinks coefficients toward zero but never eliminates them

### Key Insights:

- **Scaling matters**: Always standardize features before applying Ridge regression
- **Bias-variance tradeoff**: Higher alpha = more bias, less variance
- **No feature selection**: Ridge keeps all features; use Lasso if sparsity is needed
- **Numerical stability**: Ridge makes the normal equation always solvable

### Practical Guidelines:

1. Start with alpha in range $[10^{-4}, 10^{4}]$ on log scale
2. Use cross-validation to select optimal alpha
3. Monitor both training and test metrics to detect overfitting
4. Consider Elastic Net if you need both L1 and L2 regularization

### Next Steps:

- Implement gradient descent version for very large datasets
- Explore Lasso (L1) and Elastic Net regularization
- Try kernel Ridge regression for non-linear relationships
- Implement Bayesian Ridge regression for uncertainty quantification